In [1]:
# ============================================================
# Cell 1. Install packages
# ============================================================

%pip install -q pandas numpy requests tqdm python-dotenv openpyxl lxml beautifulsoup4 statsmodels linearmodels

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ============================================================
# Cell 2. Configuration
# ============================================================

from pathlib import Path
from getpass import getpass
import os
import json
import time
import zipfile
import re
import io
import warnings

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm
from dotenv import load_dotenv, set_key

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1) Project paths
# ------------------------------------------------------------

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
CACHE_DIR = DATA_DIR / "cache"
OUT_DIR = PROJECT_DIR / "output"
LOG_DIR = PROJECT_DIR / "logs"

for d in [DATA_DIR, RAW_DIR, CACHE_DIR, OUT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2) User file path
# ------------------------------------------------------------

KRX_FILE = Path(r"C:\Users\starw\Downloads\krx_market_list_clean.csv")

if not KRX_FILE.exists():
    raise FileNotFoundError(
        f"KRX file not found: {KRX_FILE}\n"
        "파일 경로가 다르면 KRX_FILE 값을 수정하세요."
    )

# ------------------------------------------------------------
# 3) Analysis period
# ------------------------------------------------------------
# 처음에는 2018-2024가 안정적입니다.
# Open DART 일부 정기보고서 주요정보 API는 2015년 이후 자료 제공인 경우가 많습니다.

START_YEAR = 2018
END_YEAR = 2024
YEARS = list(range(START_YEAR, END_YEAR + 1))

# ------------------------------------------------------------
# 4) Report code
# ------------------------------------------------------------
# 11011 = 사업보고서
# 11012 = 반기보고서
# 11013 = 1분기보고서
# 11014 = 3분기보고서

REPORT_CODE = "11011"

# ------------------------------------------------------------
# 5) Sample mode
# ------------------------------------------------------------
# API 호출이 많으므로 처음에는 PILOT_MODE=True로 50개 기업만 테스트하세요.
# 잘 작동하면 PILOT_MODE=False로 바꾸면 됩니다.

PILOT_MODE = True
PILOT_N_FIRMS = 50

# ------------------------------------------------------------
# 6) API rate control
# ------------------------------------------------------------
# Open DART 문서상 요청 제한 초과 오류 코드가 존재하므로 캐싱과 sleep을 같이 사용합니다.
# 일반적으로 많은 호출을 한번에 하면 020 요청 제한 초과가 날 수 있습니다.

REQUEST_SLEEP = 0.15
TIMEOUT = 30
MAX_RETRIES = 3

# ------------------------------------------------------------
# 7) API key
# ------------------------------------------------------------

ENV_PATH = PROJECT_DIR / ".env"
load_dotenv(ENV_PATH)

DART_API_KEY = os.getenv("DART_API_KEY")

if not DART_API_KEY:
    DART_API_KEY = getpass("Open DART API key를 입력하세요: ").strip()
    set_key(str(ENV_PATH), "DART_API_KEY", DART_API_KEY)
    print(f"API key saved to {ENV_PATH}")
else:
    print("DART_API_KEY loaded from .env")

BASE_URL = "https://opendart.fss.or.kr/api"

print("PROJECT_DIR:", PROJECT_DIR)
print("KRX_FILE:", KRX_FILE)
print("YEARS:", YEARS)
print("PILOT_MODE:", PILOT_MODE)

API key saved to c:\Users\starw\.vscode\practice\.env
PROJECT_DIR: c:\Users\starw\.vscode\practice
KRX_FILE: C:\Users\starw\Downloads\krx_market_list_clean.csv
YEARS: [2018, 2019, 2020, 2021, 2022, 2023, 2024]
PILOT_MODE: True


In [3]:
# ============================================================
# Cell 3. Utility functions
# ============================================================

def clean_stock_code(x):
    """
    종목코드를 6자리 문자열로 정리.
    Open DART stock_code는 일반적으로 6자리 숫자입니다.
    """
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = re.sub(r"\.0$", "", s)
    if re.fullmatch(r"\d+", s):
        return s.zfill(6)
    return s


def is_numeric_stock_code(x):
    """
    일반 보통주 중심 분석을 위해 6자리 숫자 종목코드만 남김.
    예: 0120G0 같은 비표준 코드 제거.
    """
    if pd.isna(x):
        return False
    return bool(re.fullmatch(r"\d{6}", str(x).strip()))


def parse_number(x):
    """
    Open DART의 금액/수량 문자열을 숫자로 변환.
    예: '1,234', '-', '해당사항없음', '' 등을 처리.
    """
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s in ["", "-", "—", "해당사항 없음", "해당사항없음", "nan", "None"]:
        return np.nan
    
    # 괄호 음수 처리: (1,234) -> -1234
    neg = False
    if s.startswith("(") and s.endswith(")"):
        neg = True
        s = s[1:-1]
    
    # 숫자, 소수점, 마이너스만 남김
    s = re.sub(r"[^0-9\.\-]", "", s)
    if s in ["", "-", ".", "-."]:
        return np.nan
    
    try:
        val = float(s)
        return -val if neg else val
    except:
        return np.nan


def safe_divide(a, b):
    if pd.isna(a) or pd.isna(b) or b == 0:
        return np.nan
    return a / b


def cache_path(endpoint_name, params):
    """
    API 요청별 캐시 파일 경로 생성.
    같은 요청을 다시 하지 않도록 저장합니다.
    """
    key_parts = [endpoint_name]
    for k in sorted(params.keys()):
        if k == "crtfc_key":
            continue
        key_parts.append(f"{k}-{params[k]}")
    fname = "__".join(key_parts)
    fname = re.sub(r"[^A-Za-z0-9가-힣_\-\.]", "_", fname)
    return CACHE_DIR / f"{fname}.json"


def dart_get_json(endpoint_name, params, use_cache=True):
    """
    Open DART JSON API 요청 함수.
    status 000 = 정상.
    status 013 = 조회된 데이터 없음. 이 경우 빈 list 반환.
    """
    params = dict(params)
    params["crtfc_key"] = DART_API_KEY
    
    cp = cache_path(endpoint_name, params)
    if use_cache and cp.exists():
        with open(cp, "r", encoding="utf-8") as f:
            return json.load(f)
    
    url = f"{BASE_URL}/{endpoint_name}"
    
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            time.sleep(REQUEST_SLEEP)
            r = requests.get(url, params=params, timeout=TIMEOUT)
            r.raise_for_status()
            data = r.json()
            
            if use_cache:
                with open(cp, "w", encoding="utf-8") as f:
                    json.dump(data, f, ensure_ascii=False, indent=2)
            
            return data
        
        except Exception as e:
            last_err = e
            time.sleep(1.0 * attempt)
    
    return {
        "status": "ERROR",
        "message": str(last_err),
        "list": []
    }


def dart_list(endpoint_name, params, use_cache=True):
    """
    대부분의 Open DART API는 JSON 안에 list를 반환합니다.
    """
    data = dart_get_json(endpoint_name, params, use_cache=use_cache)
    status = str(data.get("status", ""))
    
    if status == "000":
        return data.get("list", [])
    elif status == "013":
        return []
    else:
        # 요청 제한, 잘못된 키 등은 로그로 남김
        return []


def save_df(df, name):
    path = OUT_DIR / name
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Saved: {path} | shape={df.shape}")
    return path

In [4]:
# ============================================================
# Cell 4. Load KRX listed firms and build KOSPI non-financial sample
# ============================================================

krx = pd.read_csv(KRX_FILE, dtype=str, encoding="utf-8-sig")

print("Raw KRX shape:", krx.shape)
print("Columns:", krx.columns.tolist())
display(krx.head())

# 필요한 컬럼명 확인
required_cols = ["stock_code", "corp_name", "market", "sector"]
missing = [c for c in required_cols if c not in krx.columns]

if missing:
    raise ValueError(f"필수 컬럼 없음: {missing}. 현재 컬럼: {krx.columns.tolist()}")

krx["stock_code"] = krx["stock_code"].apply(clean_stock_code)
krx["market"] = krx["market"].astype(str).str.strip()
krx["sector"] = krx["sector"].astype(str).str.strip()

# 일반 6자리 숫자 종목코드만 사용
krx["is_standard_stock_code"] = krx["stock_code"].apply(is_numeric_stock_code)

# 금융업 제외
financial_keywords = [
    "금융", "은행", "보험", "증권", "투자", "신탁",
    "카드", "캐피탈", "리스", "부동산", "회사 본부"
]

krx["is_financial_auto"] = krx["sector"].apply(
    lambda x: any(k in x for k in financial_keywords)
)

# 기존 파일에 conservative flag가 있으면 우선 사용
if "is_financial_conservative" in krx.columns:
    krx["is_financial_conservative"] = krx["is_financial_conservative"].astype(str).str.lower().map({
        "true": True, "false": False, "1": True, "0": False
    }).fillna(krx["is_financial_auto"])
else:
    krx["is_financial_conservative"] = krx["is_financial_auto"]

sample = krx[
    (krx["market"] == "KOSPI") &
    (krx["is_standard_stock_code"]) &
    (~krx["is_financial_conservative"])
].copy()

sample = sample.drop_duplicates(subset=["stock_code"]).reset_index(drop=True)

print("KOSPI non-financial standard stock sample:", sample.shape)
print(sample[["stock_code", "corp_name", "market", "sector"]].head(20))

if PILOT_MODE:
    sample = sample.head(PILOT_N_FIRMS).copy()
    print(f"PILOT MODE: using first {len(sample)} firms")

save_df(sample, "01_sample_kospi_nonfinancial.csv")

Raw KRX shape: (2764, 12)
Columns: ['stock_code', 'corp_name', 'market', 'market_raw', 'sector', 'main_products', 'listing_date', 'fiscal_month', 'region', 'is_financial_narrow', 'is_financial_conservative', 'exclude_reason']


,stock_code,corp_name,market,market_raw,sector,main_products,listing_date,fiscal_month,region,is_financial_narrow,is_financial_conservative,exclude_reason
0,477850,마키나락스,KOSDAQ,코스닥,소프트웨어 개발 및 공급업,Runway Platform (산업특화 인공지능 개발 및 운영 체계 구축 플랫폼),2026-05-20,12월,서울특별시,False,False,NaN
1,288180,케이피항공산업,KOSDAQ,코스닥,"항공기,우주선 및 부품 제조업",항공기 및 우주 방산 구조물 제작,2026-05-19,12월,경상남도,False,False,NaN
2,487580,폴레드,KOSDAQ,코스닥,가정용 기기 제조업,"유아용품(카시트 및 관련 악세서리, 유아가전 등)",2026-05-14,12월,충청남도,False,False,NaN
3,439960,코스모로보틱스,KOSDAQ,코스닥,의료용 기기 제조업,웨어러블 로봇,2026-05-11,12월,서울특별시,False,False,NaN
4,0129K0,신한제18호스팩,KOSDAQ,코스닥,금융 지원 서비스업,금융 지원 서비스업,2026-04-30,12월,서울특별시,True,True,금융


KOSPI non-financial standard stock sample: (691, 14)
   stock_code  corp_name market                             sector
0      217590        티엠씨  KOSPI                      절연선 및 케이블 제조업
1      317450       명인제약  KOSPI                            의약품 제조업
2      439260       대한조선  KOSPI                        선박 및 보트 건조업
3      483650      달바글로벌  KOSPI                        기타 화학제품 제조업
4      480370     씨케이솔루션  KOSPI                      일반 목적용 기계 제조업
5      064400     LG씨엔에스  KOSPI            컴퓨터 프로그래밍, 시스템 통합 및 관리업
6      484870     엠앤씨솔루션  KOSPI                      특수 목적용 기계 제조업
7      475560      더본코리아  KOSPI                          상품 종합 도매업
8      489790       한화비전  KOSPI                     통신 및 방송 장비 제조업
9      079900     전진건설로봇  KOSPI                      특수 목적용 기계 제조업
10     062040       산일전기  KOSPI  전동기, 발전기 및 전기 변환 · 공급 · 제어 장치 제조업
11     462870       시프트업  KOSPI                     소프트웨어 개발 및 공급업
12     034230      파라다이스  KOSPI                 유원지 및 기타 오락관련 서비스업
13     44

WindowsPath('c:/Users/starw/.vscode/practice/output/01_sample_kospi_nonfinancial.csv')

In [5]:
# ============================================================
# Cell 5. Download and parse Open DART corp codes
# ============================================================

import xml.etree.ElementTree as ET

def download_corp_codes():
    """
    Open DART corpCode.xml 다운로드.
    응답은 zip 파일이며 내부에 CORPCODE.xml이 들어 있습니다.
    """
    zip_path = RAW_DIR / "corpCode.zip"
    xml_path = RAW_DIR / "CORPCODE.xml"
    
    if xml_path.exists():
        print("CORPCODE.xml already exists.")
        return xml_path
    
    url = f"{BASE_URL}/corpCode.xml"
    params = {"crtfc_key": DART_API_KEY}
    
    print("Downloading corpCode.zip...")
    r = requests.get(url, params=params, timeout=TIMEOUT)
    r.raise_for_status()
    
    with open(zip_path, "wb") as f:
        f.write(r.content)
    
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(RAW_DIR)
    
    if not xml_path.exists():
        # 내부 파일명이 다를 경우 탐색
        candidates = list(RAW_DIR.glob("*.xml"))
        if candidates:
            xml_path = candidates[0]
        else:
            raise FileNotFoundError("CORPCODE.xml not found after unzip.")
    
    return xml_path


def parse_corp_codes(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    rows = []
    for item in root.findall("list"):
        row = {}
        for child in item:
            row[child.tag] = child.text
        rows.append(row)
    
    df = pd.DataFrame(rows)
    df["stock_code"] = df["stock_code"].apply(clean_stock_code)
    return df


xml_path = download_corp_codes()
corp_codes = parse_corp_codes(xml_path)

print("corp_codes shape:", corp_codes.shape)
display(corp_codes.head())

# stock_code가 있는 상장기업만
corp_listed = corp_codes[
    corp_codes["stock_code"].apply(is_numeric_stock_code)
].copy()

# KRX sample과 매칭
sample_dart = sample.merge(
    corp_listed[["corp_code", "corp_name", "stock_code", "modify_date"]],
    on="stock_code",
    how="left",
    suffixes=("", "_dart")
)

print("Matched sample:", sample_dart.shape)
print("Missing corp_code:", sample_dart["corp_code"].isna().sum())

display(sample_dart[["stock_code", "corp_name", "corp_name_dart", "corp_code", "sector"]].head(20))

sample_dart = sample_dart.dropna(subset=["corp_code"]).copy()
save_df(sample_dart, "02_sample_with_dart_corp_code.csv")

corp_codes shape: (118220, 5)


,corp_code,corp_name,corp_eng_name,stock_code,modify_date
0,00434003,다코,Daco corporation,,20170630
1,00430964,굿앤엘에스,"Good & LS Co.,Ltd.",,20170630
2,00388953,크레디피아제이십오차유동화전문회사,Credipia 25th Asset Securitization Specialty L...,,20170630
3,00179984,연방건설산업,youn bao,,20170630
4,00420143,브룩스피알아이오토메이션잉크,"BROOKS-PRI Automation, Inc.",,20170630


Matched sample: (50, 17)
Missing corp_code: 0


,stock_code,corp_name,corp_name_dart,corp_code,sector
0,217590,티엠씨,티엠씨,00949161,절연선 및 케이블 제조업
1,317450,명인제약,명인제약,00179489,의약품 제조업
2,439260,대한조선,대한조선,00182696,선박 및 보트 건조업
3,483650,달바글로벌,달바글로벌,01649204,기타 화학제품 제조업
4,480370,씨케이솔루션,씨케이솔루션,01210677,일반 목적용 기계 제조업
5,064400,LG씨엔에스,LG씨엔에스,00139834,"컴퓨터 프로그래밍, 시스템 통합 및 관리업"
6,484870,엠앤씨솔루션,엠앤씨솔루션,01519790,특수 목적용 기계 제조업
7,475560,더본코리아,더본코리아,00968607,상품 종합 도매업
8,489790,한화비전,한화비전,01867758,통신 및 방송 장비 제조업
9,079900,전진건설로봇,전진건설로봇,00372129,특수 목적용 기계 제조업


Saved: c:\Users\starw\.vscode\practice\output\02_sample_with_dart_corp_code.csv | shape=(50, 17)


WindowsPath('c:/Users/starw/.vscode/practice/output/02_sample_with_dart_corp_code.csv')

In [6]:
# ============================================================
# Cell 6. Collect annual report filings and corrections
# ============================================================

def fetch_filings_for_firm_year(corp_code, year):
    """
    특정 기업-연도 공시목록 수집.
    pblntf_ty='A'는 정기공시 계열로 사용.
    """
    params = {
        "corp_code": corp_code,
        "bgn_de": f"{year}0101",
        "end_de": f"{year}1231",
        "page_no": 1,
        "page_count": 100,
        "pblntf_ty": "A"
    }
    
    rows = []
    while True:
        data = dart_get_json("list.json", params)
        status = str(data.get("status", ""))
        
        if status == "000":
            part = data.get("list", [])
            rows.extend(part)
            
            total_page = int(data.get("total_page", 1) or 1)
            page_no = int(params["page_no"])
            if page_no >= total_page:
                break
            params["page_no"] = page_no + 1
        
        elif status == "013":
            break
        
        else:
            # 에러 로그
            rows.append({
                "corp_code": corp_code,
                "year": year,
                "status": status,
                "message": data.get("message")
            })
            break
    
    for r in rows:
        r["query_year"] = year
    return rows


all_filings = []

for _, row in tqdm(sample_dart.iterrows(), total=len(sample_dart), desc="Fetching filings"):
    corp_code = row["corp_code"]
    for year in YEARS:
        rows = fetch_filings_for_firm_year(corp_code, year)
        all_filings.extend(rows)

filings = pd.DataFrame(all_filings)

print("filings shape:", filings.shape)
display(filings.head())

if filings.empty:
    raise ValueError("No filings collected. API key, corp_code, or request conditions may be wrong.")

# 날짜/연도 정리
filings["rcept_dt"] = filings.get("rcept_dt", pd.Series(dtype=str)).astype(str)
filings["year"] = filings["query_year"].astype(int)

# 사업보고서 관련 공시만
filings["report_nm"] = filings["report_nm"].astype(str)

annual_filings = filings[
    filings["report_nm"].str.contains("사업보고서", na=False)
].copy()

# 정정공시 식별
annual_filings["is_correction"] = annual_filings["report_nm"].str.contains("정정", na=False).astype(int)

# 사업보고서 정정공시
annual_corrections = annual_filings[
    annual_filings["is_correction"] == 1
].copy()

print("annual filings:", annual_filings.shape)
print("annual corrections:", annual_corrections.shape)

display(annual_corrections[["corp_name", "corp_code", "rcept_no", "report_nm", "rcept_dt", "year"]].head(20))

save_df(filings, "03_all_periodic_filings_raw.csv")
save_df(annual_filings, "04_annual_report_filings.csv")
save_df(annual_corrections, "05_annual_report_corrections.csv")

Fetching filings:   0%|          | 0/50 [00:00<?, ?it/s]

filings shape: (727, 10)


,corp_code,corp_name,stock_code,corp_cls,report_nm,rcept_no,flr_nm,rcept_dt,rm,query_year
0,01649204,달바글로벌,483650,Y,[기재정정]분기보고서 (2024.09),20241125000168,달바글로벌,20241125,,2024
1,01649204,달바글로벌,483650,Y,분기보고서 (2024.09),20241113000384,달바글로벌,20241113,정,2024
2,00139834,LG씨엔에스,064400,Y,분기보고서 (2018.09),20181114001092,LG씨엔에스,20181114,,2018
3,00139834,LG씨엔에스,064400,Y,반기보고서 (2018.06),20180814001258,LG씨엔에스,20180814,,2018
4,00139834,LG씨엔에스,064400,Y,분기보고서 (2018.03),20180515002040,LG씨엔에스,20180515,,2018


annual filings: (193, 12)
annual corrections: (54, 12)


,corp_name,corp_code,rcept_no,report_nm,rcept_dt,year
21,LG씨엔에스,00139834,20220405002309,[기재정정]사업보고서 (2021.12),20220405,2022
51,파라다이스,00171265,20210419000082,[기재정정]사업보고서 (2020.12),20210419,2021
75,에이피알,01190568,20180528000239,[기재정정]사업보고서 (2017.12),20180528,2018
77,에이피알,01190568,20180417000178,[기재정정]사업보고서 (2017.12),20180417,2018
82,에이피알,01190568,20191114002071,[기재정정]사업보고서 (2018.12),20191114,2019
86,에이피알,01190568,20191114001781,[기재정정]사업보고서 (2017.12),20191114,2019
89,에이피알,01190568,20190528000207,[기재정정]사업보고서 (2018.12),20190528,2019
95,에이피알,01190568,20200924000296,[기재정정]사업보고서 (2019.12),20200924,2020
99,에이피알,01190568,20200924000286,[기재정정]사업보고서 (2018.12),20200924,2020
103,에이피알,01190568,20200924000275,[기재정정]사업보고서 (2017.12),20200924,2020


Saved: c:\Users\starw\.vscode\practice\output\03_all_periodic_filings_raw.csv | shape=(727, 11)
Saved: c:\Users\starw\.vscode\practice\output\04_annual_report_filings.csv | shape=(193, 12)
Saved: c:\Users\starw\.vscode\practice\output\05_annual_report_corrections.csv | shape=(54, 12)


WindowsPath('c:/Users/starw/.vscode/practice/output/05_annual_report_corrections.csv')

In [7]:
# ============================================================
# Cell 7. Build correction variables
# ============================================================

# firm-year base
base_panel = (
    sample_dart[["stock_code", "corp_code", "corp_name", "market", "sector", "listing_date"]]
    .assign(key=1)
    .merge(pd.DataFrame({"year": YEARS, "key": 1}), on="key")
    .drop(columns="key")
)

# correction count
correction_agg = (
    annual_corrections
    .groupby(["corp_code", "year"], as_index=False)
    .agg(
        CorrectionCount=("rcept_no", "nunique"),
        FirstCorrectionDate=("rcept_dt", "min"),
        LastCorrectionDate=("rcept_dt", "max")
    )
)

panel = base_panel.merge(correction_agg, on=["corp_code", "year"], how="left")

panel["CorrectionCount"] = panel["CorrectionCount"].fillna(0).astype(int)
panel["CorrectionDummy"] = (panel["CorrectionCount"] > 0).astype(int)

print(panel["CorrectionDummy"].value_counts())
display(panel.head())

save_df(panel, "06_panel_with_correction_vars.csv")

CorrectionDummy
0    318
1     32
Name: count, dtype: int64


,stock_code,corp_code,corp_name,market,sector,listing_date,year,CorrectionCount,FirstCorrectionDate,LastCorrectionDate,CorrectionDummy
0,217590,00949161,티엠씨,KOSPI,절연선 및 케이블 제조업,2025-12-15,2018,0,NaN,NaN,0
1,217590,00949161,티엠씨,KOSPI,절연선 및 케이블 제조업,2025-12-15,2019,0,NaN,NaN,0
2,217590,00949161,티엠씨,KOSPI,절연선 및 케이블 제조업,2025-12-15,2020,0,NaN,NaN,0
3,217590,00949161,티엠씨,KOSPI,절연선 및 케이블 제조업,2025-12-15,2021,0,NaN,NaN,0
4,217590,00949161,티엠씨,KOSPI,절연선 및 케이블 제조업,2025-12-15,2022,0,NaN,NaN,0


Saved: c:\Users\starw\.vscode\practice\output\06_panel_with_correction_vars.csv | shape=(350, 11)


WindowsPath('c:/Users/starw/.vscode/practice/output/06_panel_with_correction_vars.csv')

In [8]:
# ============================================================
# Cell 8. Optional: classify material corrections using XML keyword search
# ============================================================

MATERIAL_KEYWORDS = [
    "재무제표", "연결재무제표", "손익계산서", "재무상태표",
    "매출", "영업이익", "당기순이익", "자산", "부채", "자본",
    "감사의견", "감사인", "회계감사", "감사용역", "비감사용역",
    "최대주주", "소액주주", "사외이사", "이사회",
    "특수관계", "관계기업", "종속기업", "타법인", "출자"
]

def document_cache_path(rcept_no):
    return CACHE_DIR / f"document_{rcept_no}.txt"


def fetch_document_text(rcept_no):
    """
    document.xml은 zip/xml 형태일 수 있어 텍스트로 최대한 추출합니다.
    """
    cp = document_cache_path(rcept_no)
    if cp.exists():
        return cp.read_text(encoding="utf-8", errors="ignore")
    
    url = f"{BASE_URL}/document.xml"
    params = {"crtfc_key": DART_API_KEY, "rcept_no": rcept_no}
    
    try:
        time.sleep(REQUEST_SLEEP)
        r = requests.get(url, params=params, timeout=TIMEOUT)
        r.raise_for_status()
        content = r.content
        
        text = ""
        # zip 여부 확인
        if content[:2] == b"PK":
            with zipfile.ZipFile(io.BytesIO(content)) as z:
                for name in z.namelist():
                    try:
                        text += z.read(name).decode("utf-8", errors="ignore")
                    except:
                        try:
                            text += z.read(name).decode("cp949", errors="ignore")
                        except:
                            pass
        else:
            try:
                text = content.decode("utf-8", errors="ignore")
            except:
                text = content.decode("cp949", errors="ignore")
        
        cp.write_text(text, encoding="utf-8", errors="ignore")
        return text
    
    except Exception as e:
        return ""


# 파일럿에서는 너무 오래 걸리지 않게 정정공시만 대상으로 수행
material_rows = []

for _, r in tqdm(annual_corrections.iterrows(), total=len(annual_corrections), desc="Classifying material corrections"):
    rcept_no = r["rcept_no"]
    text = fetch_document_text(rcept_no)
    hits = [kw for kw in MATERIAL_KEYWORDS if kw in text]
    
    material_rows.append({
        "corp_code": r["corp_code"],
        "corp_name": r.get("corp_name"),
        "year": int(r["year"]),
        "rcept_no": rcept_no,
        "rcept_dt": r.get("rcept_dt"),
        "report_nm": r.get("report_nm"),
        "keyword_hits": "|".join(hits),
        "n_keyword_hits": len(hits),
        "MaterialCorrection": int(len(hits) > 0)
    })

material_corrections = pd.DataFrame(material_rows)

print("material_corrections shape:", material_corrections.shape)
display(material_corrections.head(20))

material_agg = (
    material_corrections
    .groupby(["corp_code", "year"], as_index=False)
    .agg(
        MaterialCorrectionCount=("MaterialCorrection", "sum"),
        MaterialKeywordHits=("n_keyword_hits", "sum")
    )
)

panel = panel.merge(material_agg, on=["corp_code", "year"], how="left")
panel["MaterialCorrectionCount"] = panel["MaterialCorrectionCount"].fillna(0).astype(int)
panel["MaterialCorrectionDummy"] = (panel["MaterialCorrectionCount"] > 0).astype(int)
panel["MaterialKeywordHits"] = panel["MaterialKeywordHits"].fillna(0).astype(int)

save_df(material_corrections, "07_material_corrections_detail.csv")
save_df(panel, "08_panel_with_material_correction_vars.csv")

Classifying material corrections:   0%|          | 0/54 [00:00<?, ?it/s]

material_corrections shape: (54, 9)


,corp_code,corp_name,year,rcept_no,rcept_dt,report_nm,keyword_hits,n_keyword_hits,MaterialCorrection
0,00139834,LG씨엔에스,2022,20220405002309,20220405,[기재정정]사업보고서 (2021.12),,0,0
1,00171265,파라다이스,2021,20210419000082,20210419,[기재정정]사업보고서 (2020.12),,0,0
2,01190568,에이피알,2018,20180528000239,20180528,[기재정정]사업보고서 (2017.12),,0,0
3,01190568,에이피알,2018,20180417000178,20180417,[기재정정]사업보고서 (2017.12),,0,0
4,01190568,에이피알,2019,20191114002071,20191114,[기재정정]사업보고서 (2018.12),,0,0
5,01190568,에이피알,2019,20191114001781,20191114,[기재정정]사업보고서 (2017.12),,0,0
6,01190568,에이피알,2019,20190528000207,20190528,[기재정정]사업보고서 (2018.12),,0,0
7,01190568,에이피알,2020,20200924000296,20200924,[기재정정]사업보고서 (2019.12),,0,0
8,01190568,에이피알,2020,20200924000286,20200924,[기재정정]사업보고서 (2018.12),,0,0
9,01190568,에이피알,2020,20200924000275,20200924,[기재정정]사업보고서 (2017.12),,0,0


Saved: c:\Users\starw\.vscode\practice\output\07_material_corrections_detail.csv | shape=(54, 9)
Saved: c:\Users\starw\.vscode\practice\output\08_panel_with_material_correction_vars.csv | shape=(350, 14)


WindowsPath('c:/Users/starw/.vscode/practice/output/08_panel_with_material_correction_vars.csv')

In [9]:
# ============================================================
# Cell 9. Collect periodic report key information
# ============================================================

PERIODIC_ENDPOINTS = {
    "auditor_opinion": "accnutAdtorNmNdAdtOpinion.json",
    "audit_contract": "adtServcCnclsSttus.json",
    "non_audit_contract": "accnutAdtorNonAdtServcCnclsSttus.json",
    "outside_director": "outcmpnyDrctrNdChangeSttus.json",
    "largest_shareholder": "hyslrSttus.json",
    "minority_shareholder": "mrhlSttus.json",
}

def fetch_periodic_endpoint(endpoint_name, corp_code, year, reprt_code=REPORT_CODE):
    params = {
        "corp_code": corp_code,
        "bsns_year": str(year),
        "reprt_code": reprt_code
    }
    return dart_list(endpoint_name, params)


endpoint_frames = {}

for label, endpoint in PERIODIC_ENDPOINTS.items():
    rows = []
    print(f"\nCollecting {label}: {endpoint}")
    
    for _, firm in tqdm(sample_dart.iterrows(), total=len(sample_dart), desc=label):
        corp_code = firm["corp_code"]
        for year in YEARS:
            out = fetch_periodic_endpoint(endpoint, corp_code, year)
            for item in out:
                item["query_year"] = year
                item["endpoint_label"] = label
                rows.append(item)
    
    df = pd.DataFrame(rows)
    endpoint_frames[label] = df
    save_df(df, f"raw_{label}.csv")
    print(label, df.shape)
    if not df.empty:
        display(df.head())

auditor_opinion:   0%|          | 0/50 [00:00<?, ?it/s]

Saved: c:\Users\starw\.vscode\practice\output\raw_auditor_opinion.csv | shape=(663, 13)
auditor_opinion (663, 13)


,rcept_no,corp_cls,corp_code,corp_name,bsns_year,adtor,adt_opinion,adt_reprt_spcmnt_matter,stlm_dt,query_year,endpoint_label,emphs_matter,core_adt_matter
0,20250318000466,Y,01649204,달바글로벌,제9기\n(당기),삼일회계법인,적정의견,-,2024-12-31,2024,auditor_opinion,NaN,NaN
1,20250318000466,Y,01649204,달바글로벌,제9기\n(당기),삼일회계법인,적정의견,-,2024-12-31,2024,auditor_opinion,NaN,NaN
2,20250318000466,Y,01649204,달바글로벌,제8기\n(전기),태성회계법인,적정의견,-,2024-12-31,2024,auditor_opinion,(주1),NaN
3,20250318000466,Y,01649204,달바글로벌,제8기\n(전기),태성회계법인,적정의견,-,2024-12-31,2024,auditor_opinion,(주2),NaN
4,20250318000466,Y,01649204,달바글로벌,제7기\n(전전기),태성회계법인,적정의견,-,2024-12-31,2024,auditor_opinion,NaN,NaN


audit_contract:   0%|          | 0/50 [00:00<?, ?it/s]

Saved: c:\Users\starw\.vscode\practice\output\raw_audit_contract.csv | shape=(519, 16)
audit_contract (519, 16)


...

Saved: c:\Users\starw\.vscode\practice\output\raw_largest_shareholder.csv | shape=(1716, 15)
largest_shareholder (1716, 15)


,rcept_no,corp_cls,corp_code,corp_name,stock_knd,nm,relate,bsis_posesn_stock_co,bsis_posesn_stock_qota_rt,trmend_posesn_stock_co,trmend_posesn_stock_qota_rt,rm,stlm_dt,query_year,endpoint_label
0,20250318000466,Y,01649204,달바글로벌,보통주,반성연,본인,"1,605,645",14.30,"1,994,780",17.40,(주1),2024-12-31,2024,largest_shareholder
1,20250318000466,Y,01649204,달바글로벌,보통주,유명한,등기임원,0,0.00,"122,437",1.10,(주2),2024-12-31,2024,largest_shareholder
2,20250318000466,Y,01649204,달바글로벌,보통주,계,NaN,"1,605,645",14.30,"2,117,217",18.50,-,2024-12-31,2024,largest_shareholder
3,20250318000466,Y,01649204,달바글로벌,우선주,계,NaN,-,-,-,-,-,2024-12-31,2024,largest_shareholder
4,20250320001576,Y,01210677,씨케이솔루션,보통주,김유곤,최대주주 본인,"50,001",36.80,"2,529,650",26.81,주3),2024-12-31,2024,largest_shareholder


minority_shareholder:   0%|          | 0/50 [00:00<?, ?it/s]

Saved: c:\Users\starw\.vscode\practice\output\raw_minority_shareholder.csv | shape=(173, 14)
minority_shareholder (173, 14)


,rcept_no,corp_cls,corp_code,corp_name,se,shrholdr_co,shrholdr_tot_co,shrholdr_rate,hold_stock_co,stock_tot_co,hold_stock_rate,stlm_dt,query_year,endpoint_label
0,20250318000466,Y,01649204,달바글로벌,소액주주,76,96,79.2%,"1,711,442","11,465,665",14.9%,2024-12-31,2024,minority_shareholder
1,20250320001576,Y,01210677,씨케이솔루션,소액주주,13,21,61.90%,"690,400","9,434,861",7.32%,2024-12-31,2024,minority_shareholder
2,20190401004162,Y,00139834,LG씨엔에스,소액주주,"4,999",-,99.91%,"10,448,503",-,11.97%,2018-12-31,2018,minority_shareholder
3,20200330003747,Y,00139834,LG씨엔에스,소액주주,"5,086",-,99.91%,"10,517,344",-,12.05%,2019-12-31,2019,minority_shareholder
4,20210331001781,Y,00139834,LG씨엔에스,소액주주,"5,376","5,381",99.9%,"10,596,073","87,197,353",12.14%,2020-12-31,2020,minority_shareholder


In [10]:
# ============================================================
# Cell 10. Process audit and governance variables
# ============================================================

def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


# ------------------------------------------------------------
# 10-1. Auditor opinion
# ------------------------------------------------------------
auditor_df = endpoint_frames.get("auditor_opinion", pd.DataFrame()).copy()

if not auditor_df.empty:
    auditor_df["year"] = auditor_df["query_year"].astype(int)
    
    adtor_col = first_existing_col(auditor_df, ["adtor", "auditor", "nm"])
    opinion_col = first_existing_col(auditor_df, ["adt_opinion", "opinion", "auditor_opinion"])
    
    keep_cols = ["corp_code", "year"]
    if adtor_col: keep_cols.append(adtor_col)
    if opinion_col: keep_cols.append(opinion_col)
    
    auditor_vars = auditor_df[keep_cols].drop_duplicates(["corp_code", "year"]).copy()
    
    if adtor_col:
        auditor_vars = auditor_vars.rename(columns={adtor_col: "AuditorName"})
        big4_keywords = ["삼일", "삼정", "한영", "안진", "PWC", "PwC", "KPMG", "EY", "Deloitte"]
        auditor_vars["Big4Auditor"] = auditor_vars["AuditorName"].astype(str).apply(
            lambda x: int(any(k.lower() in x.lower() for k in big4_keywords))
        )
    
    if opinion_col:
        auditor_vars = auditor_vars.rename(columns={opinion_col: "AuditOpinion"})
        auditor_vars["CleanOpinion"] = auditor_vars["AuditOpinion"].astype(str).str.contains("적정", na=False).astype(int)
else:
    auditor_vars = pd.DataFrame(columns=["corp_code", "year"])

print("auditor_vars:", auditor_vars.shape)
display(auditor_vars.head())


# ------------------------------------------------------------
# 10-2. Audit contract fees
# ------------------------------------------------------------
audit_contract = endpoint_frames.get("audit_contract", pd.DataFrame()).copy()

if not audit_contract.empty:
    audit_contract["year"] = audit_contract["query_year"].astype(int)
    
    fee_cols = [
        "mendng",
        "adt_cntrct_dtls_mendng",
        "audit_fee",
        "cntrct_mendng"
    ]
    fee_col = first_existing_col(audit_contract, fee_cols)
    
    time_cols = [
        "tot_reqre_time",
        "adt_cntrct_dtls_time"
    ]
    time_col = first_existing_col(audit_contract, time_cols)
    
    audit_contract["AuditFee_raw"] = audit_contract[fee_col] if fee_col else np.nan
    audit_contract["AuditHours_raw"] = audit_contract[time_col] if time_col else np.nan
    
    audit_contract["AuditFee"] = audit_contract["AuditFee_raw"].apply(parse_number)
    audit_contract["AuditHours"] = audit_contract["AuditHours_raw"].apply(parse_number)
    
    audit_fee_vars = (
        audit_contract
        .groupby(["corp_code", "year"], as_index=False)
        .agg(
            AuditFee=("AuditFee", "max"),
            AuditHours=("AuditHours", "max")
        )
    )
else:
    audit_fee_vars = pd.DataFrame(columns=["corp_code", "year", "AuditFee", "AuditHours"])

print("audit_fee_vars:", audit_fee_vars.shape)
display(audit_fee_vars.head())


# ------------------------------------------------------------
# 10-3. Non-audit service contracts
# ------------------------------------------------------------
non_audit = endpoint_frames.get("non_audit_contract", pd.DataFrame()).copy()

if not non_audit.empty:
    non_audit["year"] = non_audit["query_year"].astype(int)
    
    # 비감사용역 금액 컬럼은 문서/시점에 따라 다르게 나올 수 있어 후보를 넓게 둠.
    amount_candidates = [
        "mendng",
        "cntrct_mendng",
        "srv_mendng",
        "contract_amount",
        "fee",
        "rm"
    ]
    amount_col = first_existing_col(non_audit, amount_candidates)
    
    if amount_col:
        non_audit["NonAuditFee"] = non_audit[amount_col].apply(parse_number)
    else:
        # 금액 컬럼을 못 찾으면 행 수만 사용
        non_audit["NonAuditFee"] = np.nan
    
    non_audit_vars = (
        non_audit
        .groupby(["corp_code", "year"], as_index=False)
        .agg(
            NonAuditContractCount=("corp_code", "size"),
            NonAuditFee=("NonAuditFee", "sum")
        )
    )
else:
    non_audit_vars = pd.DataFrame(columns=["corp_code", "year", "NonAuditContractCount", "NonAuditFee"])

print("non_audit_vars:", non_audit_vars.shape)
display(non_audit_vars.head())


# ------------------------------------------------------------
# 10-4. Outside directors
# ------------------------------------------------------------
outside = endpoint_frames.get("outside_director", pd.DataFrame()).copy()

if not outside.empty:
    outside["year"] = outside["query_year"].astype(int)
    
    for c in ["drctr_co", "otcmp_drctr_co", "apnt", "rlsofc", "mdstrm_resig"]:
        if c in outside.columns:
            outside[c] = outside[c].apply(parse_number)
        else:
            outside[c] = np.nan
    
    outside_vars = (
        outside
        .groupby(["corp_code", "year"], as_index=False)
        .agg(
            BoardSize=("drctr_co", "max"),
            OutsideDirectorCount=("otcmp_drctr_co", "max"),
            OutsideDirectorAppointed=("apnt", "max"),
            OutsideDirectorDismissed=("rlsofc", "max"),
            OutsideDirectorResigned=("mdstrm_resig", "max")
        )
    )
    
    outside_vars["OutsideDirectorRatio"] = outside_vars.apply(
        lambda r: safe_divide(r["OutsideDirectorCount"], r["BoardSize"]),
        axis=1
    )
else:
    outside_vars = pd.DataFrame(columns=[
        "corp_code", "year", "BoardSize", "OutsideDirectorCount", "OutsideDirectorRatio"
    ])

print("outside_vars:", outside_vars.shape)
display(outside_vars.head())


# ------------------------------------------------------------
# 10-5. Largest shareholder
# ------------------------------------------------------------
largest = endpoint_frames.get("largest_shareholder", pd.DataFrame()).copy()

if not largest.empty:
    largest["year"] = largest["query_year"].astype(int)
    
    # 최대주주 지분율 후보 컬럼
    ratio_candidates = ["trmend_posesn_stock_qota_rt", "posesn_stock_qota_rt", "qota_rt", "rt"]
    ratio_col = first_existing_col(largest, ratio_candidates)
    
    if ratio_col:
        largest["LargestShareholderOwnership"] = largest[ratio_col].apply(parse_number)
    else:
        largest["LargestShareholderOwnership"] = np.nan
    
    largest_vars = (
        largest
        .groupby(["corp_code", "year"], as_index=False)
        .agg(
            LargestShareholderOwnership=("LargestShareholderOwnership", "max")
        )
    )
else:
    largest_vars = pd.DataFrame(columns=["corp_code", "year", "LargestShareholderOwnership"])

print("largest_vars:", largest_vars.shape)
display(largest_vars.head())


# ------------------------------------------------------------
# 10-6. Minority shareholders
# ------------------------------------------------------------
minority = endpoint_frames.get("minority_shareholder", pd.DataFrame()).copy()

if not minority.empty:
    minority["year"] = minority["query_year"].astype(int)
    
    # 소액주주 비율 후보
    ratio_candidates = ["mrhl_rate", "shrholdr_rate", "qota_rt", "rt"]
    count_candidates = ["mrhl_co", "shrholdr_co", "stockholdr_co"]
    
    ratio_col = first_existing_col(minority, ratio_candidates)
    count_col = first_existing_col(minority, count_candidates)
    
    minority["MinorityShareholderRatio"] = minority[ratio_col].apply(parse_number) if ratio_col else np.nan
    minority["MinorityShareholderCount"] = minority[count_col].apply(parse_number) if count_col else np.nan
    
    minority_vars = (
        minority
        .groupby(["corp_code", "year"], as_index=False)
        .agg(
            MinorityShareholderRatio=("MinorityShareholderRatio", "max"),
            MinorityShareholderCount=("MinorityShareholderCount", "max")
        )
    )
else:
    minority_vars = pd.DataFrame(columns=["corp_code", "year", "MinorityShareholderRatio", "MinorityShareholderCount"])

print("minority_vars:", minority_vars.shape)
display(minority_vars.head())

auditor_vars: (173, 6)


,corp_code,year,AuditorName,AuditOpinion,Big4Auditor,CleanOpinion
0,01649204,2024,삼일회계법인,적정의견,1,1
6,01210677,2024,인덕회계법인,적정의견,0,0
12,00139834,2018,안진 회계법인,적정,1,1
15,00139834,2019,안진회계법인,적정,1,1
18,00139834,2020,안진 회계법인,적정,1,1


audit_fee_vars: (173, 4)


,corp_code,year,AuditFee,AuditHours
0,00108612,2023,NaN,NaN
1,00108612,2024,NaN,NaN
2,00139834,2018,280.0,4326.0
3,00139834,2019,360.0,4326.0
4,00139834,2020,NaN,NaN


non_audit_vars: (173, 4)


...

,corp_code,year,BoardSize,OutsideDirectorCount,OutsideDirectorAppointed,OutsideDirectorDismissed,OutsideDirectorResigned,OutsideDirectorRatio
0,00108612,2023,8.0,3.0,3.0,NaN,NaN,0.375000
1,00108612,2024,7.0,3.0,1.0,NaN,1.0,0.428571
2,00139834,2020,5.0,0.0,NaN,NaN,NaN,0.000000
3,00139834,2021,5.0,0.0,NaN,NaN,NaN,0.000000
4,00139834,2022,5.0,NaN,NaN,NaN,NaN,NaN


largest_vars: (173, 3)


,corp_code,year,LargestShareholderOwnership
0,00108612,2023,40.64
1,00108612,2024,39.96
2,00139834,2018,87.30
3,00139834,2019,87.30
4,00139834,2020,52.33


minority_vars: (173, 4)


,corp_code,year,MinorityShareholderRatio,MinorityShareholderCount
0,00108612,2023,99.98,68372.0
1,00108612,2024,99.99,65355.0
2,00139834,2018,99.91,4999.0
3,00139834,2019,99.91,5086.0
4,00139834,2020,99.90,5376.0


In [11]:
# ============================================================
# Cell 11. Collect financial statement key accounts
# ============================================================

def fetch_financial_accounts(corp_code, year, reprt_code=REPORT_CODE):
    params = {
        "corp_code": corp_code,
        "bsns_year": str(year),
        "reprt_code": reprt_code
    }
    return dart_list("fnlttSinglAcnt.json", params)


fin_rows = []

for _, firm in tqdm(sample_dart.iterrows(), total=len(sample_dart), desc="financial accounts"):
    corp_code = firm["corp_code"]
    for year in YEARS:
        out = fetch_financial_accounts(corp_code, year)
        for item in out:
            item["query_year"] = year
            fin_rows.append(item)

fin_raw = pd.DataFrame(fin_rows)

print("fin_raw:", fin_raw.shape)
display(fin_raw.head())

save_df(fin_raw, "raw_financial_accounts.csv")


def make_financial_vars(fin_raw):
    if fin_raw.empty:
        return pd.DataFrame(columns=[
            "corp_code", "year", "Assets", "Liabilities", "Equity",
            "Revenue", "OperatingIncome", "NetIncome"
        ])
    
    df = fin_raw.copy()
    df["year"] = df["query_year"].astype(int)
    df["amount"] = df["thstrm_amount"].apply(parse_number)
    df["account_nm"] = df["account_nm"].astype(str)
    
    # 연결재무제표 우선
    if "fs_div" in df.columns:
        df["fs_priority"] = np.where(df["fs_div"].eq("CFS"), 1, 0)
    else:
        df["fs_priority"] = 0
    
    account_map = {
        "Assets": ["자산총계"],
        "Liabilities": ["부채총계"],
        "Equity": ["자본총계"],
        "Revenue": ["매출액", "영업수익"],
        "OperatingIncome": ["영업이익"],
        "NetIncome": ["당기순이익", "분기순이익", "반기순이익"]
    }
    
    out = df[["corp_code", "year"]].drop_duplicates().copy()
    
    for var, names in account_map.items():
        tmp = df[df["account_nm"].isin(names)].copy()
        if tmp.empty:
            out[var] = np.nan
            continue
        
        # 연결 우선, 중복이면 큰 금액 우선
        tmp = tmp.sort_values(["corp_code", "year", "fs_priority"], ascending=[True, True, False])
        tmp = tmp.groupby(["corp_code", "year"], as_index=False)["amount"].first()
        tmp = tmp.rename(columns={"amount": var})
        out = out.merge(tmp, on=["corp_code", "year"], how="left")
    
    out["Size"] = np.log(out["Assets"].replace(0, np.nan))
    out["Leverage"] = out.apply(lambda r: safe_divide(r["Liabilities"], r["Assets"]), axis=1)
    out["ROA"] = out.apply(lambda r: safe_divide(r["NetIncome"], r["Assets"]), axis=1)
    out["Loss"] = (out["NetIncome"] < 0).astype(float)
    
    return out


fin_vars = make_financial_vars(fin_raw)

print("fin_vars:", fin_vars.shape)
display(fin_vars.head())

save_df(fin_vars, "financial_vars.csv")

financial accounts:   0%|          | 0/50 [00:00<?, ?it/s]

fin_raw: (4146, 22)


,rcept_no,reprt_code,bsns_year,corp_code,stock_code,fs_div,fs_nm,sj_div,sj_nm,account_nm,...,thstrm_amount,frmtrm_nm,frmtrm_dt,frmtrm_amount,bfefrmtrm_nm,bfefrmtrm_dt,bfefrmtrm_amount,ord,currency,query_year
0,20250318000466,11011,2024,01649204,483650,CFS,연결재무제표,BS,재무상태표,유동자산,...,"128,308,613,752",제 8 기,2023.12.31 현재,"73,534,860,990",제 7 기,2022.12.31 현재,"36,367,952,162",1,KRW,2024
1,20250318000466,11011,2024,01649204,483650,CFS,연결재무제표,BS,재무상태표,비유동자산,...,"7,799,551,441",제 8 기,2023.12.31 현재,"4,468,748,136",제 7 기,2022.12.31 현재,"2,987,618,289",3,KRW,2024
2,20250318000466,11011,2024,01649204,483650,CFS,연결재무제표,BS,재무상태표,자산총계,...,"136,108,165,193",제 8 기,2023.12.31 현재,"78,003,609,126",제 7 기,2022.12.31 현재,"39,355,570,451",5,KRW,2024
3,20250318000466,11011,2024,01649204,483650,CFS,연결재무제표,BS,재무상태표,유동부채,...,"31,320,324,507",제 8 기,2023.12.31 현재,"56,037,891,135",제 7 기,2022.12.31 현재,"34,280,975,281",7,KRW,2024
4,20250318000466,11011,2024,01649204,483650,CFS,연결재무제표,BS,재무상태표,비유동부채,...,"2,718,768,216",제 8 기,2023.12.31 현재,"6,451,975,264",제 7 기,2022.12.31 현재,"3,382,533,584",9,KRW,2024


Saved: c:\Users\starw\.vscode\practice\output\raw_financial_accounts.csv | shape=(4146, 22)
fin_vars: (158, 12)


,corp_code,year,Assets,Liabilities,Equity,Revenue,OperatingIncome,NetIncome,Size,Leverage,ROA,Loss
0,01649204,2024,1.361082e+11,3.403909e+10,1.020691e+11,3.090626e+11,5.984471e+10,NaN,25.636716,0.250089,NaN,0.0
1,01210677,2024,1.884231e+11,9.729188e+10,9.113119e+10,2.957725e+11,1.652431e+10,NaN,25.961956,0.516348,NaN,0.0
2,00139834,2023,4.040680e+12,2.172423e+12,1.868256e+12,5.605300e+12,4.640484e+11,NaN,29.027434,0.537638,NaN,0.0
3,00139834,2024,4.504508e+12,2.381697e+12,2.122810e+12,5.982627e+12,5.128641e+11,NaN,29.136100,0.528736,NaN,0.0
4,01519790,2024,3.571133e+11,1.917275e+11,1.653858e+11,2.827966e+11,3.482492e+10,NaN,26.601319,0.536881,NaN,0.0


Saved: c:\Users\starw\.vscode\practice\output\financial_vars.csv | shape=(158, 12)


WindowsPath('c:/Users/starw/.vscode/practice/output/financial_vars.csv')

In [12]:
# ============================================================
# Cell 12. Merge final firm-year panel
# ============================================================

final_panel = panel.copy()

merge_dfs = [
    auditor_vars,
    audit_fee_vars,
    non_audit_vars,
    outside_vars,
    largest_vars,
    minority_vars,
    fin_vars
]

for df in merge_dfs:
    if df is not None and not df.empty:
        final_panel = final_panel.merge(df, on=["corp_code", "year"], how="left")

# 파생변수
final_panel["NonAuditFeeRatio"] = final_panel.apply(
    lambda r: safe_divide(r.get("NonAuditFee", np.nan), r.get("AuditFee", np.nan)),
    axis=1
)

final_panel["LogAuditFee"] = np.log(final_panel["AuditFee"].replace(0, np.nan))
final_panel["LogAssets"] = final_panel["Size"]
final_panel["CorrectionCount_Log"] = np.log1p(final_panel["CorrectionCount"])

if "MaterialCorrectionDummy" not in final_panel.columns:
    final_panel["MaterialCorrectionDummy"] = np.nan
if "MaterialCorrectionCount" not in final_panel.columns:
    final_panel["MaterialCorrectionCount"] = np.nan

# 보기 좋은 컬럼 순서
front_cols = [
    "stock_code", "corp_code", "corp_name", "year", "market", "sector",
    "CorrectionDummy", "CorrectionCount", "MaterialCorrectionDummy", "MaterialCorrectionCount",
    "AuditorName", "Big4Auditor", "CleanOpinion",
    "AuditFee", "NonAuditFee", "NonAuditFeeRatio",
    "BoardSize", "OutsideDirectorCount", "OutsideDirectorRatio",
    "LargestShareholderOwnership",
    "MinorityShareholderRatio", "MinorityShareholderCount",
    "Assets", "Liabilities", "Equity", "Revenue", "OperatingIncome", "NetIncome",
    "LogAssets", "Leverage", "ROA", "Loss"
]

existing_front = [c for c in front_cols if c in final_panel.columns]
other_cols = [c for c in final_panel.columns if c not in existing_front]
final_panel = final_panel[existing_front + other_cols].copy()

print("final_panel:", final_panel.shape)
display(final_panel.head())

save_df(final_panel, "final_dart_correction_panel.csv")

final_panel: (350, 45)


,stock_code,corp_code,corp_name,year,market,sector,CorrectionDummy,CorrectionCount,MaterialCorrectionDummy,MaterialCorrectionCount,...,MaterialKeywordHits,AuditOpinion,AuditHours,NonAuditContractCount,OutsideDirectorAppointed,OutsideDirectorDismissed,OutsideDirectorResigned,Size,LogAuditFee,CorrectionCount_Log
0,217590,00949161,티엠씨,2018,KOSPI,절연선 및 케이블 제조업,0,0,0,0,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,217590,00949161,티엠씨,2019,KOSPI,절연선 및 케이블 제조업,0,0,0,0,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,217590,00949161,티엠씨,2020,KOSPI,절연선 및 케이블 제조업,0,0,0,0,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,217590,00949161,티엠씨,2021,KOSPI,절연선 및 케이블 제조업,0,0,0,0,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,217590,00949161,티엠씨,2022,KOSPI,절연선 및 케이블 제조업,0,0,0,0,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


Saved: c:\Users\starw\.vscode\practice\output\final_dart_correction_panel.csv | shape=(350, 45)


WindowsPath('c:/Users/starw/.vscode/practice/output/final_dart_correction_panel.csv')

In [13]:
# ============================================================
# Cell 13. Descriptive statistics
# ============================================================

desc_vars = [
    "CorrectionDummy", "CorrectionCount", "MaterialCorrectionDummy",
    "NonAuditFeeRatio", "LogAuditFee", "Big4Auditor", "CleanOpinion",
    "OutsideDirectorRatio", "LargestShareholderOwnership",
    "MinorityShareholderRatio", "LogAssets", "Leverage", "ROA", "Loss"
]

desc_vars = [v for v in desc_vars if v in final_panel.columns]

desc = final_panel[desc_vars].describe().T
display(desc)

save_df(desc.reset_index().rename(columns={"index": "variable"}), "table_01_descriptive_statistics.csv")

# 연도별 정정공시 비율
yearly_corr = (
    final_panel
    .groupby("year", as_index=False)
    .agg(
        n_firm_years=("corp_code", "count"),
        correction_rate=("CorrectionDummy", "mean"),
        correction_count_mean=("CorrectionCount", "mean")
    )
)

display(yearly_corr)
save_df(yearly_corr, "table_02_yearly_correction_rate.csv")

# 업종별 정정공시 비율
sector_corr = (
    final_panel
    .groupby("sector", as_index=False)
    .agg(
        n_firm_years=("corp_code", "count"),
        correction_rate=("CorrectionDummy", "mean"),
        correction_count_mean=("CorrectionCount", "mean")
    )
    .sort_values("n_firm_years", ascending=False)
)

display(sector_corr.head(30))
save_df(sector_corr, "table_03_sector_correction_rate.csv")

,count,mean,std,min,25%,50%,75%,max
CorrectionDummy,350.0,0.091429,0.288630,0.000000,0.000000,0.000000,0.000000,1.000000
CorrectionCount,350.0,0.154286,0.628065,0.000000,0.000000,0.000000,0.000000,7.000000
MaterialCorrectionDummy,350.0,0.040000,0.196240,0.000000,0.000000,0.000000,0.000000,1.000000
NonAuditFeeRatio,25.0,-0.001668,0.008333,-0.041667,0.000000,0.000000,0.000000,0.000000
LogAuditFee,25.0,9.929873,6.465885,4.094345,5.634790,11.225243,12.013701,36.736001
Big4Auditor,173.0,0.728324,0.446115,0.000000,0.000000,1.000000,1.000000,1.000000
CleanOpinion,173.0,0.907514,0.290551,0.000000,1.000000,1.000000,1.000000,1.000000
OutsideDirectorRatio,143.0,0.469156,0.176260,0.000000,0.375000,0.500000,0.571429,1.500000
LargestShareholderOwnership,173.0,50.188960,18.754630,18.500000,35.700000,46.270000,66.250000,100.000000
MinorityShareholderRatio,162.0,98.856432,6.877160,26.360000,99.920000,99.980000,99.990000,100.000000


Saved: c:\Users\starw\.vscode\practice\output\table_01_descriptive_statistics.csv | shape=(14, 9)


,year,n_firm_years,correction_rate,correction_count_mean
0,2018,50,0.12,0.18
1,2019,50,0.08,0.24
2,2020,50,0.04,0.16
3,2021,50,0.08,0.14
4,2022,50,0.06,0.06
5,2023,50,0.12,0.12
6,2024,50,0.14,0.18


Saved: c:\Users\starw\.vscode\practice\output\table_02_yearly_correction_rate.csv | shape=(7, 4)


,sector,n_firm_years,correction_rate,correction_count_mean
0,1차 철강 제조업,35,0.057143,0.057143
7,기타 화학제품 제조업,21,0.142857,0.476190
20,일차전지 및 이차전지 제조업,21,0.142857,0.142857
30,특수 목적용 기계 제조업,21,0.000000,0.000000
3,"건축기술, 엔지니어링 및 관련 기술 서비스업",14,0.214286,0.285714
14,운송장비 임대업,14,0.357143,0.500000
16,의료용 기기 제조업,14,0.142857,0.142857
5,기초 화학물질 제조업,14,0.071429,0.071429
12,소프트웨어 개발 및 공급업,14,0.214286,0.857143
11,선박 및 보트 건조업,14,0.000000,0.000000


Saved: c:\Users\starw\.vscode\practice\output\table_03_sector_correction_rate.csv | shape=(33, 4)


WindowsPath('c:/Users/starw/.vscode/practice/output/table_03_sector_correction_rate.csv')

In [23]:

# ============================================================
# Cell 14. Main regression: CorrectionDummy
# ============================================================

import statsmodels.formula.api as smf

# ----------------------------------------------------------
# 진단: 어떤 변수가 실제로 데이터를 갖고 있는지 먼저 확인
# ----------------------------------------------------------
candidate_rhs = [
    "NonAuditFeeRatio",
    "OutsideDirectorRatio",
    "LargestShareholderOwnership",
    "LogAssets",
    "Leverage",
    "ROA",
    "Loss",
]

print("=== final_panel 변수별 비결측치 수 ===")
diag_cols = ["CorrectionDummy"] + candidate_rhs
for col in diag_cols:
    if col in final_panel.columns:
        n_valid = final_panel[col].replace([np.inf, -np.inf], np.nan).notna().sum()
        print(f"  {col:40s}: {n_valid:5d} / {len(final_panel)}")
    else:
        print(f"  {col:40s}: 컬럼 없음")

active_rhs = [
    c for c in candidate_rhs
    if c in final_panel.columns and final_panel[c].replace([np.inf, -np.inf], np.nan).notna().any()
]
print(f"\nActive RHS vars ({len(active_rhs)}): {active_rhs}")

if len(active_rhs) == 0:
    print(
        "\n[SKIP] 회귀에 사용할 수 있는 RHS 변수가 없습니다.\n"
        "원인: Cell 9(거버넌스) 또는 Cell 11(재무제표) API 응답이 비어 있습니다.\n"
        "조치: 해당 셀을 다시 실행하거나 DART API 키와 corp_code를 확인하세요.\n"
        "fin_vars shape: " + str(fin_vars.shape if 'fin_vars' in dir() else 'N/A')
    )
    model = None
else:
    reg_df = final_panel.copy()
    reg_df = reg_df.replace([np.inf, -np.inf], np.nan)

    keep_cols = ["CorrectionDummy"] + active_rhs + ["year", "sector", "corp_code"]
    reg_df = reg_df[[c for c in keep_cols if c in reg_df.columns]].copy()
    reg_df = reg_df.dropna(subset=["CorrectionDummy"] + active_rhs)

    if "sector" in reg_df.columns:
        reg_df = reg_df[reg_df["sector"].notna() & (reg_df["sector"].astype(str).str.strip() != "")]
    if "year" in reg_df.columns:
        reg_df = reg_df[reg_df["year"].notna()]

    print(f"\nRegression data shape: {reg_df.shape}")

    if reg_df.empty:
        print("[SKIP] dropna 후 관측치 없음 — 회귀를 건너뜁니다.")
        model = None
    else:
        year_levels   = reg_df["year"].nunique()   if "year"   in reg_df.columns else 0
        sector_levels = reg_df["sector"].nunique() if "sector" in reg_df.columns else 0

        rhs_terms = list(active_rhs)
        if year_levels   > 1: rhs_terms.append("C(year)")
        if sector_levels > 1: rhs_terms.append("C(sector)")

        formula = "CorrectionDummy ~ " + " + ".join(rhs_terms)
        print("Formula:", formula)

        model = smf.ols(formula=formula, data=reg_df).fit(
            cov_type="cluster",
            cov_kwds={"groups": reg_df["corp_code"]}
        )
        print(model.summary())

        with open(OUT_DIR / "regression_01_correction_dummy_ols.txt", "w", encoding="utf-8") as f:
            f.write(model.summary().as_text())


=== final_panel 변수별 비결측치 수 ===
  CorrectionDummy                         :   350 / 350
  NonAuditFeeRatio                        :    25 / 350
  OutsideDirectorRatio                    :   143 / 350
  LargestShareholderOwnership             :   173 / 350
  LogAssets                               :   158 / 350
  Leverage                                :   158 / 350
  ROA                                     :     0 / 350
  Loss                                    :   158 / 350

Active RHS vars (6): ['NonAuditFeeRatio', 'OutsideDirectorRatio', 'LargestShareholderOwnership', 'LogAssets', 'Leverage', 'Loss']

Regression data shape: (0, 10)
[SKIP] dropna 후 관측치 없음 — 회귀를 건너뜁니다.


In [24]:

# ============================================================
# Cell 15. Count regression: CorrectionCount
# ============================================================

candidate_rhs_c = [
    "NonAuditFeeRatio",
    "OutsideDirectorRatio",
    "LargestShareholderOwnership",
    "LogAssets",
    "Leverage",
    "ROA",
    "Loss",
]

active_rhs_c = [
    c for c in candidate_rhs_c
    if c in final_panel.columns and final_panel[c].replace([np.inf, -np.inf], np.nan).notna().any()
]
print(f"Active RHS vars ({len(active_rhs_c)}): {active_rhs_c}")

if len(active_rhs_c) == 0:
    print(
        "\n[SKIP] 회귀에 사용할 수 있는 RHS 변수가 없습니다.\n"
        "Cell 9(거버넌스) 또는 Cell 11(재무제표)를 먼저 실행하세요."
    )
    poisson_model = None
else:
    count_df = final_panel.copy()
    count_df = count_df.replace([np.inf, -np.inf], np.nan)

    keep_cols_c = ["CorrectionCount"] + active_rhs_c + ["year", "sector", "corp_code"]
    count_df = count_df[[c for c in keep_cols_c if c in count_df.columns]].copy()
    count_df = count_df.dropna(subset=["CorrectionCount"] + active_rhs_c)

    if "sector" in count_df.columns:
        count_df = count_df[count_df["sector"].notna() & (count_df["sector"].astype(str).str.strip() != "")]
    if "year" in count_df.columns:
        count_df = count_df[count_df["year"].notna()]

    print(f"Count regression data shape: {count_df.shape}")

    if count_df.empty:
        print("[SKIP] dropna 후 관측치 없음 — Poisson 회귀를 건너뜁니다.")
        poisson_model = None
    else:
        year_levels   = count_df["year"].nunique()   if "year"   in count_df.columns else 0
        sector_levels = count_df["sector"].nunique() if "sector" in count_df.columns else 0

        rhs_terms_count = list(active_rhs_c)
        if year_levels   > 1: rhs_terms_count.append("C(year)")
        if sector_levels > 1: rhs_terms_count.append("C(sector)")

        formula_count = "CorrectionCount ~ " + " + ".join(rhs_terms_count)
        print("Formula:", formula_count)

        poisson_model = smf.poisson(formula=formula_count, data=count_df).fit(maxiter=200, disp=False)
        print(poisson_model.summary())

        with open(OUT_DIR / "regression_02_correction_count_poisson.txt", "w", encoding="utf-8") as f:
            f.write(poisson_model.summary().as_text())


Active RHS vars (6): ['NonAuditFeeRatio', 'OutsideDirectorRatio', 'LargestShareholderOwnership', 'LogAssets', 'Leverage', 'Loss']
Count regression data shape: (0, 10)
[SKIP] dropna 후 관측치 없음 — Poisson 회귀를 건너뜁니다.


In [25]:

# ============================================================
# Cell 16. Material correction regression
# ============================================================

if "MaterialCorrectionDummy" not in final_panel.columns or final_panel["MaterialCorrectionDummy"].notna().sum() == 0:
    print("MaterialCorrectionDummy is not available. Run Cell 8 first.")
    mat_model = None
else:
    candidate_rhs_m = [
        "NonAuditFeeRatio",
        "OutsideDirectorRatio",
        "LargestShareholderOwnership",
        "LogAssets",
        "Leverage",
        "ROA",
        "Loss",
    ]

    active_rhs_m = [
        c for c in candidate_rhs_m
        if c in final_panel.columns and final_panel[c].replace([np.inf, -np.inf], np.nan).notna().any()
    ]
    print(f"Active RHS vars ({len(active_rhs_m)}): {active_rhs_m}")

    if len(active_rhs_m) == 0:
        print(
            "\n[SKIP] 회귀에 사용할 수 있는 RHS 변수가 없습니다.\n"
            "Cell 9(거버넌스) 또는 Cell 11(재무제표)를 먼저 실행하세요."
        )
        mat_model = None
    else:
        mat_df = final_panel.copy()
        mat_df = mat_df.replace([np.inf, -np.inf], np.nan)

        keep_cols_m = ["MaterialCorrectionDummy"] + active_rhs_m + ["year", "sector", "corp_code"]
        mat_df = mat_df[[c for c in keep_cols_m if c in mat_df.columns]].copy()
        mat_df = mat_df.dropna(subset=["MaterialCorrectionDummy"] + active_rhs_m)

        if "sector" in mat_df.columns:
            mat_df = mat_df[mat_df["sector"].notna() & (mat_df["sector"].astype(str).str.strip() != "")]
        if "year" in mat_df.columns:
            mat_df = mat_df[mat_df["year"].notna()]

        print(f"Material regression data shape: {mat_df.shape}")

        if mat_df.empty:
            print("[SKIP] dropna 후 관측치 없음 — 회귀를 건너뜁니다.")
            mat_model = None
        else:
            year_levels   = mat_df["year"].nunique()   if "year"   in mat_df.columns else 0
            sector_levels = mat_df["sector"].nunique() if "sector" in mat_df.columns else 0

            rhs_terms_mat = list(active_rhs_m)
            if year_levels   > 1: rhs_terms_mat.append("C(year)")
            if sector_levels > 1: rhs_terms_mat.append("C(sector)")

            formula_mat = "MaterialCorrectionDummy ~ " + " + ".join(rhs_terms_mat)
            print("Formula:", formula_mat)

            mat_model = smf.ols(formula=formula_mat, data=mat_df).fit(
                cov_type="cluster",
                cov_kwds={"groups": mat_df["corp_code"]}
            )
            print(mat_model.summary())

            with open(OUT_DIR / "regression_03_material_correction_dummy_ols.txt", "w", encoding="utf-8") as f:
                f.write(mat_model.summary().as_text())


Active RHS vars (6): ['NonAuditFeeRatio', 'OutsideDirectorRatio', 'LargestShareholderOwnership', 'LogAssets', 'Leverage', 'Loss']
Material regression data shape: (0, 10)
[SKIP] dropna 후 관측치 없음 — 회귀를 건너뜁니다.


In [26]:
# ============================================================
# Cell 17. Final summary for ChatGPT interpretation
# ============================================================

summary = {}

summary["sample"] = {
    "n_firms": int(final_panel["corp_code"].nunique()),
    "n_firm_years": int(len(final_panel)),
    "start_year": int(final_panel["year"].min()),
    "end_year": int(final_panel["year"].max()),
    "correction_dummy_mean": float(final_panel["CorrectionDummy"].mean()),
    "correction_count_mean": float(final_panel["CorrectionCount"].mean()),
}

if "MaterialCorrectionDummy" in final_panel.columns:
    summary["sample"]["material_correction_dummy_mean"] = (
        None if final_panel["MaterialCorrectionDummy"].dropna().empty
        else float(final_panel["MaterialCorrectionDummy"].mean())
    )

summary["missing_rates"] = {
    c: float(final_panel[c].isna().mean())
    for c in [
        "NonAuditFeeRatio",
        "AuditFee",
        "OutsideDirectorRatio",
        "LargestShareholderOwnership",
        "MinorityShareholderRatio",
        "Assets",
        "ROA"
    ]
    if c in final_panel.columns
}

summary["yearly_correction_rate"] = yearly_corr.to_dict(orient="records")

print("===== COPY FROM HERE =====")
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("===== COPY TO HERE =====")

with open(OUT_DIR / "final_summary_for_chatgpt.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"Saved summary to {OUT_DIR / 'final_summary_for_chatgpt.json'}")

===== COPY FROM HERE =====
{
  "sample": {
    "n_firms": 50,
    "n_firm_years": 350,
    "start_year": 2018,
    "end_year": 2024,
    "correction_dummy_mean": 0.09142857142857143,
    "correction_count_mean": 0.15428571428571428,
    "material_correction_dummy_mean": 0.04
  },
  "missing_rates": {
    "NonAuditFeeRatio": 0.9285714285714286,
    "AuditFee": 0.9285714285714286,
    "OutsideDirectorRatio": 0.5914285714285714,
    "LargestShareholderOwnership": 0.5057142857142857,
    "MinorityShareholderRatio": 0.5371428571428571,
    "Assets": 0.5485714285714286,
    "ROA": 1.0
  },
  "yearly_correction_rate": [
    {
      "year": 2018,
      "n_firm_years": 50,
      "correction_rate": 0.12,
      "correction_count_mean": 0.18
    },
    {
      "year": 2019,
      "n_firm_years": 50,
      "correction_rate": 0.08,
      "correction_count_mean": 0.24
    },
    {
      "year": 2020,
      "n_firm_years": 50,
      "correction_rate": 0.04,
      "correction_count_mean": 0.16
    },
 

In [27]:
# ============================================================
# Cell 18. Check outputs
# ============================================================

print("Output files:")
for p in sorted(OUT_DIR.glob("*")):
    print(p.name)

Output files:
01_sample_kospi_nonfinancial.csv
02_sample_with_dart_corp_code.csv
03_all_periodic_filings_raw.csv
04_annual_report_filings.csv
05_annual_report_corrections.csv
06_panel_with_correction_vars.csv
07_material_corrections_detail.csv
08_panel_with_material_correction_vars.csv
final_dart_correction_panel.csv
final_summary_for_chatgpt.json
financial_vars.csv
raw_audit_contract.csv
raw_auditor_opinion.csv
raw_financial_accounts.csv
raw_largest_shareholder.csv
raw_minority_shareholder.csv
raw_non_audit_contract.csv
raw_outside_director.csv
table_01_descriptive_statistics.csv
table_02_yearly_correction_rate.csv
table_03_sector_correction_rate.csv


In [29]:
# ============================================================
# Final Cell. Collect key results for ChatGPT interpretation
# ============================================================

import json
import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 80)
print("DART CORRECTION STUDY: FINAL RESULT SUMMARY")
print("=" * 80)

# ------------------------------------------------------------
# 1. Check required objects
# ------------------------------------------------------------

required_objects = ["final_panel"]
missing_objects = []

for obj in required_objects:
    if obj not in globals():
        missing_objects.append(obj)

if missing_objects:
    raise ValueError(f"Missing objects: {missing_objects}. 먼저 앞 셀들을 실행하세요.")

# ------------------------------------------------------------
# 2. Basic sample information
# ------------------------------------------------------------

print("\n[1] SAMPLE INFORMATION")
print("-" * 80)

sample_info = {
    "number_of_firms": final_panel["corp_code"].nunique() if "corp_code" in final_panel.columns else None,
    "number_of_firm_years": len(final_panel),
    "start_year": int(final_panel["year"].min()) if "year" in final_panel.columns else None,
    "end_year": int(final_panel["year"].max()) if "year" in final_panel.columns else None,
    "number_of_sectors": final_panel["sector"].nunique() if "sector" in final_panel.columns else None,
}

for k, v in sample_info.items():
    print(f"{k}: {v}")

# ------------------------------------------------------------
# 3. Correction variables summary
# ------------------------------------------------------------

print("\n[2] CORRECTION VARIABLES")
print("-" * 80)

correction_vars = [
    "CorrectionDummy",
    "CorrectionCount",
    "MaterialCorrectionDummy",
    "MaterialCorrectionCount",
    "MaterialKeywordHits"
]

for var in correction_vars:
    if var in final_panel.columns:
        s = final_panel[var]
        print(f"\n{var}")
        print(f"  non-missing: {s.notna().sum()}")
        print(f"  mean:        {s.mean():.4f}" if pd.api.types.is_numeric_dtype(s) else "  mean: N/A")
        print(f"  median:      {s.median():.4f}" if pd.api.types.is_numeric_dtype(s) else "  median: N/A")
        print(f"  std:         {s.std():.4f}" if pd.api.types.is_numeric_dtype(s) else "  std: N/A")
        print(f"  min:         {s.min()}" if pd.api.types.is_numeric_dtype(s) else "  min: N/A")
        print(f"  max:         {s.max()}" if pd.api.types.is_numeric_dtype(s) else "  max: N/A")
        print("  value counts:")
        print(s.value_counts(dropna=False).head(10))

# ------------------------------------------------------------
# 4. Key explanatory variables summary
# ------------------------------------------------------------

print("\n[3] KEY EXPLANATORY VARIABLES")
print("-" * 80)

key_vars = [
    "NonAuditFeeRatio",
    "AuditFee",
    "NonAuditFee",
    "LogAuditFee",
    "Big4Auditor",
    "CleanOpinion",
    "OutsideDirectorRatio",
    "BoardSize",
    "OutsideDirectorCount",
    "LargestShareholderOwnership",
    "MinorityShareholderRatio",
    "MinorityShareholderCount",
    "Assets",
    "LogAssets",
    "Leverage",
    "ROA",
    "Loss"
]

existing_key_vars = [v for v in key_vars if v in final_panel.columns]

if existing_key_vars:
    desc = final_panel[existing_key_vars].describe().T
    display(desc)
    print(desc.to_string())
else:
    print("No key explanatory variables found.")

# ------------------------------------------------------------
# 5. Missing rates
# ------------------------------------------------------------

print("\n[4] MISSING RATES")
print("-" * 80)

missing_rates = {}

for var in existing_key_vars + [v for v in correction_vars if v in final_panel.columns]:
    missing_rates[var] = final_panel[var].isna().mean()

missing_df = (
    pd.DataFrame.from_dict(missing_rates, orient="index", columns=["missing_rate"])
    .sort_values("missing_rate", ascending=False)
)

display(missing_df)
print(missing_df.to_string())

# ------------------------------------------------------------
# 6. Yearly correction pattern
# ------------------------------------------------------------

print("\n[5] YEARLY CORRECTION PATTERN")
print("-" * 80)

if "CorrectionDummy" in final_panel.columns and "CorrectionCount" in final_panel.columns:
    yearly_summary = (
        final_panel
        .groupby("year", as_index=False)
        .agg(
            n_firm_years=("corp_code", "count"),
            n_firms=("corp_code", "nunique"),
            correction_rate=("CorrectionDummy", "mean"),
            avg_correction_count=("CorrectionCount", "mean"),
            total_corrections=("CorrectionCount", "sum")
        )
    )
    
    display(yearly_summary)
    print(yearly_summary.to_string(index=False))
else:
    yearly_summary = pd.DataFrame()
    print("Correction variables not found.")

# ------------------------------------------------------------
# 7. Sector correction pattern
# ------------------------------------------------------------

print("\n[6] SECTOR CORRECTION PATTERN")
print("-" * 80)

if "sector" in final_panel.columns and "CorrectionDummy" in final_panel.columns:
    sector_summary = (
        final_panel
        .groupby("sector", as_index=False)
        .agg(
            n_firm_years=("corp_code", "count"),
            n_firms=("corp_code", "nunique"),
            correction_rate=("CorrectionDummy", "mean"),
            avg_correction_count=("CorrectionCount", "mean")
        )
        .sort_values(["n_firm_years", "correction_rate"], ascending=[False, False])
    )
    
    display(sector_summary.head(30))
    print(sector_summary.head(30).to_string(index=False))
else:
    sector_summary = pd.DataFrame()
    print("Sector or correction variables not found.")

# ------------------------------------------------------------
# 8. Correlation matrix
# ------------------------------------------------------------

print("\n[7] CORRELATION MATRIX")
print("-" * 80)

corr_vars = [
    "CorrectionDummy",
    "CorrectionCount",
    "MaterialCorrectionDummy",
    "NonAuditFeeRatio",
    "LogAuditFee",
    "Big4Auditor",
    "OutsideDirectorRatio",
    "LargestShareholderOwnership",
    "MinorityShareholderRatio",
    "LogAssets",
    "Leverage",
    "ROA",
    "Loss"
]

corr_vars = [v for v in corr_vars if v in final_panel.columns]

if len(corr_vars) >= 2:
    corr_df = final_panel[corr_vars].replace([np.inf, -np.inf], np.nan).corr()
    display(corr_df)
    print(corr_df.to_string())
else:
    corr_df = pd.DataFrame()
    print("Not enough variables for correlation matrix.")

# ------------------------------------------------------------
# 9. Regression summaries
# ------------------------------------------------------------

print("\n[8] REGRESSION RESULTS")
print("-" * 80)

regression_objects = {
    "OLS CorrectionDummy": "model",
    "Poisson CorrectionCount": "poisson_model",
    "OLS MaterialCorrectionDummy": "mat_model"
}

for label, obj_name in regression_objects.items():
    print("\n" + "=" * 80)
    print(label)
    print("=" * 80)
    
    if obj_name in globals() and globals()[obj_name] is not None:
        reg_obj = globals()[obj_name]
        print(reg_obj.summary())
    else:
        print(f"[SKIPPED] {label} 회귀가 실행되지 않았거나 데이터 부족으로 건너뜀.")

# ------------------------------------------------------------
# 10. Coefficient table for easier interpretation
# ------------------------------------------------------------

print("\n[9] CLEAN COEFFICIENT TABLE")
print("-" * 80)

coef_tables = []

for label, obj_name in regression_objects.items():
    if obj_name in globals() and globals()[obj_name] is not None:
        reg_obj = globals()[obj_name]
        tmp = pd.DataFrame({
            "model": label,
            "variable": reg_obj.params.index,
            "coef": reg_obj.params.values,
            "std_err": reg_obj.bse.values,
            "t_or_z": reg_obj.tvalues.values,
            "p_value": reg_obj.pvalues.values
        })
        coef_tables.append(tmp)

if coef_tables:
    coef_table = pd.concat(coef_tables, ignore_index=True)
    display(coef_table)
    print(coef_table.to_string(index=False))
else:
    coef_table = pd.DataFrame()
    print("No coefficient tables available (회귀가 실행되지 않음).")

# ------------------------------------------------------------
# 11. Save everything to output files
# ------------------------------------------------------------

OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)

summary_dict = {
    "sample_info": sample_info,
    "missing_rates": missing_rates,
}

with open(OUT_DIR / "copy_paste_summary_for_chatgpt.json", "w", encoding="utf-8") as f:
    json.dump(summary_dict, f, ensure_ascii=False, indent=2)

if "yearly_summary" in locals() and not yearly_summary.empty:
    yearly_summary.to_csv(OUT_DIR / "copy_paste_yearly_summary.csv", index=False, encoding="utf-8-sig")

if "sector_summary" in locals() and not sector_summary.empty:
    sector_summary.to_csv(OUT_DIR / "copy_paste_sector_summary.csv", index=False, encoding="utf-8-sig")

if "missing_df" in locals() and not missing_df.empty:
    missing_df.to_csv(OUT_DIR / "copy_paste_missing_rates.csv", encoding="utf-8-sig")

if "corr_df" in locals() and not corr_df.empty:
    corr_df.to_csv(OUT_DIR / "copy_paste_correlation_matrix.csv", encoding="utf-8-sig")

if "coef_table" in locals() and not coef_table.empty:
    coef_table.to_csv(OUT_DIR / "copy_paste_regression_coefficients.csv", index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 12. Compact final block for copy-paste
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COMPACT COPY-PASTE BLOCK FOR CHATGPT")
print("=" * 80)

compact = {
    "sample_info": sample_info,
    "correction_summary": {},
    "missing_rates": missing_rates,
}

for var in correction_vars:
    if var in final_panel.columns and pd.api.types.is_numeric_dtype(final_panel[var]):
        compact["correction_summary"][var] = {
            "non_missing": int(final_panel[var].notna().sum()),
            "mean": None if pd.isna(final_panel[var].mean()) else float(final_panel[var].mean()),
            "median": None if pd.isna(final_panel[var].median()) else float(final_panel[var].median()),
            "std": None if pd.isna(final_panel[var].std()) else float(final_panel[var].std()),
            "min": None if pd.isna(final_panel[var].min()) else float(final_panel[var].min()),
            "max": None if pd.isna(final_panel[var].max()) else float(final_panel[var].max()),
        }

if "yearly_summary" in locals() and not yearly_summary.empty:
    compact["yearly_summary"] = yearly_summary.to_dict(orient="records")

if "coef_table" in locals() and not coef_table.empty:
    # 핵심 변수만 축약
    important_terms = [
        "NonAuditFeeRatio",
        "OutsideDirectorRatio",
        "LargestShareholderOwnership",
        "LogAssets",
        "Leverage",
        "ROA",
        "Loss"
    ]
    compact["regression_key_coefficients"] = (
        coef_table[coef_table["variable"].isin(important_terms)]
        .to_dict(orient="records")
    )

print(json.dumps(compact, ensure_ascii=False, indent=2))

print("\n" + "=" * 80)
print("END OF FINAL RESULT SUMMARY")
print("=" * 80)

print("\n저장된 파일:")
for p in sorted(OUT_DIR.glob("copy_paste_*")):
    print(" -", p)


DART CORRECTION STUDY: FINAL RESULT SUMMARY

[1] SAMPLE INFORMATION
--------------------------------------------------------------------------------
number_of_firms: 50
number_of_firm_years: 350
start_year: 2018
end_year: 2024
number_of_sectors: 33

[2] CORRECTION VARIABLES
--------------------------------------------------------------------------------

CorrectionDummy
  non-missing: 350
  mean:        0.0914
  median:      0.0000
  std:         0.2886
  min:         0
  max:         1
  value counts:
CorrectionDummy
0    318
1     32
Name: count, dtype: int64

CorrectionCount
  non-missing: 350
  mean:        0.1543
  median:      0.0000
  std:         0.6281
  min:         0
  max:         7
  value counts:
CorrectionCount
0    318
1     21
2      6
3      3
5      1
7      1
Name: count, dtype: int64

MaterialCorrectionDummy
  non-missing: 350
  mean:        0.0400
  median:      0.0000
  std:         0.1962
  min:         0
  max:         1
  value counts:
MaterialCorrectionDummy


,count,mean,std,min,25%,50%,75%,max
NonAuditFeeRatio,25.0,-1.668078e-03,8.333042e-03,-4.166667e-02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
AuditFee,25.0,3.600000e+14,1.800000e+15,6.000000e+01,2.800000e+02,7.500000e+04,1.650000e+05,9.000000e+15
NonAuditFee,173.0,-2.297038e+05,2.165327e+06,-2.019202e+07,0.000000e+00,0.000000e+00,0.000000e+00,2.151180e+05
LogAuditFee,25.0,9.929873e+00,6.465885e+00,4.094345e+00,5.634790e+00,1.122524e+01,1.201370e+01,3.673600e+01
Big4Auditor,173.0,7.283237e-01,4.461151e-01,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
CleanOpinion,173.0,9.075145e-01,2.905511e-01,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
OutsideDirectorRatio,143.0,4.691558e-01,1.762596e-01,0.000000e+00,3.750000e-01,5.000000e-01,5.714286e-01,1.500000e+00
BoardSize,147.0,5.659864e+00,1.519267e+00,2.000000e+00,5.000000e+00,6.000000e+00,7.000000e+00,1.000000e+01
OutsideDirectorCount,143.0,2.657343e+00,1.048842e+00,0.000000e+00,2.000000e+00,3.000000e+00,3.000000e+00,5.000000e+00
LargestShareholderOwnership,173.0,5.018896e+01,1.875463e+01,1.850000e+01,3.570000e+01,4.627000e+01,6.625000e+01,1.000000e+02


                             count          mean           std           min           25%           50%           75%           max
NonAuditFeeRatio              25.0 -1.668078e-03  8.333042e-03 -4.166667e-02  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
AuditFee                      25.0  3.600000e+14  1.800000e+15  6.000000e+01  2.800000e+02  7.500000e+04  1.650000e+05  9.000000e+15
NonAuditFee                  173.0 -2.297038e+05  2.165327e+06 -2.019202e+07  0.000000e+00  0.000000e+00  0.000000e+00  2.151180e+05
LogAuditFee                   25.0  9.929873e+00  6.465885e+00  4.094345e+00  5.634790e+00  1.122524e+01  1.201370e+01  3.673600e+01
Big4Auditor                  173.0  7.283237e-01  4.461151e-01  0.000000e+00  0.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00
CleanOpinion                 173.0  9.075145e-01  2.905511e-01  0.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00
OutsideDirectorRatio         143.0  4.691558e-01  1.762596e-01  0.000

,missing_rate
ROA,1.000000
NonAuditFeeRatio,0.928571
LogAuditFee,0.928571
AuditFee,0.928571
OutsideDirectorRatio,0.591429
OutsideDirectorCount,0.591429
BoardSize,0.580000
Assets,0.548571
Leverage,0.548571
Loss,0.548571


                             missing_rate
ROA                              1.000000
NonAuditFeeRatio                 0.928571
LogAuditFee                      0.928571
AuditFee                         0.928571
OutsideDirectorRatio             0.591429
OutsideDirectorCount             0.591429
BoardSize                        0.580000
Assets                           0.548571
Leverage                         0.548571
Loss                             0.548571
LogAssets                        0.548571
MinorityShareholderCount         0.537143
MinorityShareholderRatio         0.537143
NonAuditFee                      0.505714
CleanOpinion                     0.505714
Big4Auditor                      0.505714
LargestShareholderOwnership      0.505714
CorrectionDummy                  0.000000
CorrectionCount                  0.000000
MaterialCorrectionDummy          0.000000
MaterialCorrectionCount          0.000000
MaterialKeywordHits              0.000000

[5] YEARLY CORRECTION PATTERN
---

...

 year  n_firm_years  n_firms  correction_rate  avg_correction_count  total_corrections
 2018            50       50             0.12                  0.18                  9
 2019            50       50             0.08                  0.24                 12
 2020            50       50             0.04                  0.16                  8
 2021            50       50             0.08                  0.14                  7
 2022            50       50             0.06                  0.06                  3
 2023            50       50             0.12                  0.12                  6
 2024            50       50             0.14                  0.18                  9

[6] SECTOR CORRECTION PATTERN
--------------------------------------------------------------------------------


,sector,n_firm_years,n_firms,correction_rate,avg_correction_count
0,1차 철강 제조업,35,5,0.057143,0.057143
7,기타 화학제품 제조업,21,3,0.142857,0.476190
20,일차전지 및 이차전지 제조업,21,3,0.142857,0.142857
30,특수 목적용 기계 제조업,21,3,0.000000,0.000000
14,운송장비 임대업,14,2,0.357143,0.500000
3,"건축기술, 엔지니어링 및 관련 기술 서비스업",14,2,0.214286,0.285714
12,소프트웨어 개발 및 공급업,14,2,0.214286,0.857143
16,의료용 기기 제조업,14,2,0.142857,0.142857
5,기초 화학물질 제조업,14,2,0.071429,0.071429
11,선박 및 보트 건조업,14,2,0.000000,0.000000


                           sector  n_firm_years  n_firms  correction_rate  avg_correction_count
                        1차 철강 제조업            35        5         0.057143              0.057143
                      기타 화학제품 제조업            21        3         0.142857              0.476190
                  일차전지 및 이차전지 제조업            21        3         0.142857              0.142857
                    특수 목적용 기계 제조업            21        3         0.000000              0.000000
                         운송장비 임대업            14        2         0.357143              0.500000
         건축기술, 엔지니어링 및 관련 기술 서비스업            14        2         0.214286              0.285714
                   소프트웨어 개발 및 공급업            14        2         0.214286              0.857143
                       의료용 기기 제조업            14        2         0.142857              0.142857
                      기초 화학물질 제조업            14        2         0.071429              0.071429
                      선박 및 보트 건조업       

,CorrectionDummy,CorrectionCount,MaterialCorrectionDummy,NonAuditFeeRatio,LogAuditFee,Big4Auditor,OutsideDirectorRatio,LargestShareholderOwnership,MinorityShareholderRatio,LogAssets,Leverage,ROA,Loss
CorrectionDummy,1.000000,0.775498,0.643477,0.166461,0.204450,0.090150,-0.034139,0.034131,0.064794,0.012827,0.180343,NaN,NaN
CorrectionCount,0.775498,1.000000,0.321750,0.113070,0.240377,0.070245,-0.056543,0.008495,0.037508,-0.080973,0.159665,NaN,NaN
MaterialCorrectionDummy,0.643477,0.321750,1.000000,NaN,NaN,-0.009364,0.020531,0.095835,0.049658,0.081385,0.104061,NaN,NaN
NonAuditFeeRatio,0.166461,0.113070,NaN,1.000000,0.130239,-0.089166,NaN,-0.484746,-0.081213,NaN,NaN,NaN,NaN
LogAuditFee,0.204450,0.240377,NaN,0.130239,1.000000,0.359857,NaN,-0.026882,-0.184390,0.149696,-0.262390,NaN,NaN
Big4Auditor,0.090150,0.070245,-0.009364,-0.089166,0.359857,1.000000,-0.029794,0.093290,0.139606,0.223630,-0.091528,NaN,NaN
OutsideDirectorRatio,-0.034139,-0.056543,0.020531,NaN,NaN,-0.029794,1.000000,-0.130892,0.072233,0.376689,0.049302,NaN,NaN
LargestShareholderOwnership,0.034131,0.008495,0.095835,-0.484746,-0.026882,0.093290,-0.130892,1.000000,-0.124446,0.239872,-0.044503,NaN,NaN
MinorityShareholderRatio,0.064794,0.037508,0.049658,-0.081213,-0.184390,0.139606,0.072233,-0.124446,1.000000,0.141080,0.129376,NaN,NaN
LogAssets,0.012827,-0.080973,0.081385,NaN,0.149696,0.223630,0.376689,0.239872,0.141080,1.000000,0.203511,NaN,NaN


                             CorrectionDummy  CorrectionCount  MaterialCorrectionDummy  NonAuditFeeRatio  LogAuditFee  Big4Auditor  OutsideDirectorRatio  LargestShareholderOwnership  MinorityShareholderRatio  LogAssets  Leverage  ROA  Loss
CorrectionDummy                     1.000000         0.775498                 0.643477          0.166461     0.204450     0.090150             -0.034139                     0.034131                  0.064794   0.012827  0.180343  NaN   NaN
CorrectionCount                     0.775498         1.000000                 0.321750          0.113070     0.240377     0.070245             -0.056543                     0.008495                  0.037508  -0.080973  0.159665  NaN   NaN
MaterialCorrectionDummy             0.643477         0.321750                 1.000000               NaN          NaN    -0.009364              0.020531                     0.095835                  0.049658   0.081385  0.104061  NaN   NaN
NonAuditFeeRatio                    0.16